**PHASE 1: DATA PREPROCESSING - Text Cleaning**

Purpose: Clean abstracts for TF-IDF vectorization  
Input: arxiv_metadata_features.pkl  
Output: arxiv_text_cleaned.pkl  
Cleaning: Lowercase, remove special chars, normalize whitespace  
Columns Kept: id, title, abstract_clean, year, categories, metadata  
ML Involved: None - Text preprocessing  
Runtime: ~10-15 minutes  
Run Once: Never need to run again

In [1]:
# imports

import pandas as pd
import re
from tqdm import tqdm
import os

# enable progress bar for pandas

tqdm.pandas()

In [2]:
# load metadata

df = pd.read_pickle('data/processed/arxiv_metadata_features.pkl')
print(f"Loaded: {len(df):,} papers")
print(f"Columns: {list(df.columns)}")

Loaded: 2,384,622 papers
Columns: ['id', 'title', 'abstract', 'year', 'primary_category', 'all_categories', 'top_level_domain', 'num_categories', 'is_multi_category', 'has_journal', 'num_authors', 'abstract_length', 'title_length']


In [ ]:
import re
import pandas as pd
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import nltk

# Download required NLTK data (run once)
# Uncomment these lines on first run:
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Get base English stopwords
base_stopwords = set(stopwords.words('english'))

# Add custom stopwords for scientific papers
# These are common in academic writing but don't add semantic value
custom_stopwords = {
    #custom_stopwords = {
    # Meta-discussion of the paper itself (pure boilerplate)
    'paper', 'study', 'work', 'research',
    'present', 'presents', 'presented',
    'propose', 'proposed', 'proposing',
    'show', 'shows', 'shown',
    'discuss', 'discussed',
    'describe', 'described',
    'report', 'reported',
    'investigate', 'investigated',
    'consider', 'considered', 'demonstrate', 'introduce', 'compare', 
    'apply', 'observe', 
    'include', 'suggest', 'reveal', 'confirm', 'highlight',
    'indicate', 'examine', 'verify', 'imply', 'argue', 'focus',
    'motivate', 'facilitate', 'ensure',
    
    # Generic outcomes
    'result', 'results',
    'finding', 'findings',
    'conclusion', 'conclusions',
    
    # Generic process words
    'using', 'used', 'use', 'uses',
    'based',
    'provide', 'provides', 'provided',
    'allow', 'allows', 'allowed',
    'enable', 'enables', 'enabled',
    'way', 'need', 'follow', 'certain', 'fully', 'able', 
    'satisfy', 'additionally', 'help', 'especially', 'typically',
    'likely', 'recent', 'finally', 'previous', 'generally',
    'particularly', 'specifically', 'significantly', 'effectively',
    'simultaneously', 'respectively', 'perspective', 'concern',
    'regard', 'purpose',
    
    # Generic actions
    'make', 'makes', 'made', 'making',
    'give', 'gives', 'given', 'giving',
    'take', 'takes', 'taken', 'taking',
    'find', 'finds', 'found', 'finding',
    'obtain', 'obtained', 'obtaining',
    
    # Generic qualifiers
    'however', 'moreover', 'furthermore', 'therefore', 'thus', 'hence',
    'also', 'well',
    'may', 'might', 'could', 'would', 'should',
    'one', 'two', 'three', 'first', 'second', 'third',
    'different', 'various', 'several', 'many', 'much',
    'large', 'small', 'high', 'low',
    'good', 'better', 'best',
    'important', 'significant', 'main', 'major',

    # Web/URL artifacts
    'http', 'github', 'com',
    
    # LaTeX artifacts (remove)
    'mathbb', 'mathbf', 'mathrm', 'mathcal', 'mathit',
    'textit', 'textbf', 'textrm', 'emph',
    'cite', 'ref', 'label', 'eq', 'eqn', 'fig',
    'section', 'subsection', 'chapter',
    'left', 'right', 'begin', 'end', 'item', 'infty', 
    'geq', 'leq', 'frac', 'bar', 'sim',
    
    # Et al
    'et', 'al', 'etc', 'ie', 'eg', 'non'
    
    # Spelled numbers
    'zero', 'one', 'two', 'three', 'four', 'five',
    'six', 'seven', 'eight', 'nine', 'ten'
}

# Combine stopwords
all_stopwords = base_stopwords.union(custom_stopwords)

# Pattern to detect LaTeX-like terms
latex_pattern = re.compile(r'^(math|text|emph|cite|ref|label|fig|eq|section|subsection|chapter|left|right|begin|end)[a-z]*$')



In [8]:
def clean_text(text):

    """
    Enhanced text cleaning with lemmatization (gentle approach)
    
    Steps:
    1. Lowercase
    2. Remove special characters (keep only letters and spaces)
    3. Remove extra whitespace
    4. Tokenize (split into words)
    5. Remove stopwords (both standard and custom)
    6. Remove very short words (< 3 characters)
    7. Remove LaTeX artifacts
    8. Lemmatize words (objects → object, running → run, BUT preserves word integrity)
    9. Rejoin into cleaned text
    """
    if pd.isna(text) or text == '':
        return ""
    
    # Convert to string and lowercase
    text = str(text).lower()
    
    # Remove special characters, keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize
    words = text.split()
    
    # Filter and lemmatize
    cleaned_words = []
    for word in words:
        # Skip if stopword
        if word in all_stopwords:
            continue
        
        # Skip if too short (likely not meaningful)
        if len(word) < 3:
            continue
            
        # Skip if LaTeX pattern
        if latex_pattern.match(word):
            continue
        
        # Lemmatize as noun first (handles plurals: objects → object)
        lemma_noun = lemmatizer.lemmatize(word, pos='n')
        
        # Lemmatize as verb (handles: running → run, computed → compute)
        lemma_verb = lemmatizer.lemmatize(lemma_noun, pos='v')
        
        # Skip if lemmatized word is now too short or is a stopword
        if len(lemma_verb) >= 3 and lemma_verb not in all_stopwords:
            cleaned_words.append(lemma_verb)
    
    # Rejoin
    return ' '.join(cleaned_words)


# test on one abstract

sample_text = df['abstract'].iloc[0]
print("Original:")
print(sample_text[:200])
print("\nCleaned:")
print(clean_text(sample_text)[:200])

Original:
  A fully differential calculation in perturbative quantum chromodynamics is
presented for the production of massive photon pairs at hadron colliders. All
next-to-leading order perturbative contributi

Cleaned:
differential calculation perturbative quantum chromodynamics production massive photon pair hadron collider next lead order perturbative contribution quark antiquark gluon anti quark gluon gluon subpr


In [9]:
test_texts = [
    # Test 1: LaTeX artifacts
    "We present a novel method using mathbb{R} and textit{quantum} fields.",
    
    # Test 2: Generic terms
    "In this paper, we propose a new approach. The proposed method shows better results.",
    
    # Test 3: Singular/plural (should preserve word quality)
    "We study graphs and networks. The graph shows interesting properties. Multiple objects interact.",
    
    # Test 4: Real abstract sample (should preserve scientific terms)
    "A fully differential calculation in perturbative quantum chromodynamics is presented for the production of massive photon pairs.",
    
    # Test 5: Verb forms
    "We computed results using optimization. The algorithm computes optimal solutions.",
    
    # Test 6: Scientific terminology preservation
    "Observational data from galaxies reveals gravitational lensing effects in cosmological simulations."
]

for i, text in enumerate(test_texts, 1):
    print(f"\nTest {i}:")
    print(f"Original: {text}")
    print(f"Cleaned:  {clean_text(text)}")


Test 1:
Original: We present a novel method using mathbb{R} and textit{quantum} fields.
Cleaned:  novel method quantum field

Test 2:
Original: In this paper, we propose a new approach. The proposed method shows better results.
Cleaned:  new approach method

Test 3:
Original: We study graphs and networks. The graph shows interesting properties. Multiple objects interact.
Cleaned:  graph network graph interest property multiple object interact

Test 4:
Original: A fully differential calculation in perturbative quantum chromodynamics is presented for the production of massive photon pairs.
Cleaned:  differential calculation perturbative quantum chromodynamics production massive photon pair

Test 5:
Original: We computed results using optimization. The algorithm computes optimal solutions.
Cleaned:  compute optimization algorithm compute optimal solution

Test 6:
Original: Observational data from galaxies reveals gravitational lensing effects in cosmological simulations.
Cleaned:  observ

In [10]:
# clean abstracts (this could take ~10-15 minutes)

print("Cleaning abstracts...")
df['abstract_clean'] = df['abstract'].progress_apply(clean_text)

# check results

print(f"\nOriginal abstract length: {df['abstract_length'].mean():.0f} chars")
print(f"Cleaned abstract length: {df['abstract_clean'].str.len().mean():.0f} chars")
print(f"Empty abstracts: {(df['abstract_clean'] == '').sum():,}")

Cleaning abstracts...


100%|██████████| 2384622/2384622 [08:32<00:00, 4648.70it/s]



Original abstract length: 1020 chars
Cleaned abstract length: 635 chars
Empty abstracts: 5


In [11]:
# remove papers with empty abstracts after cleaning

df_before = len(df)
df = df[df['abstract_clean'] != ''].reset_index(drop=True)
df_after = len(df)

print(f"Removed {df_before - df_after:,} papers with no abstract")
print(f"Remaining: {df_after:,} papers")

Removed 5 papers with no abstract
Remaining: 2,384,617 papers


In [12]:
# save cleaned data

df.to_pickle('data/processed/arxiv_text_cleaned.pkl')
print(f"✓ Saved to: data/processed/arxiv_text_cleaned.pkl")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

✓ Saved to: data/processed/arxiv_text_cleaned.pkl
Memory: 4.92 GB


In [13]:
# verify

pickle_path = 'data/processed/arxiv_text_cleaned.pkl'

if os.path.exists(pickle_path):
    size_gb = os.path.getsize(pickle_path) / 1024**3
    print(f"✓✓✓ Success! ✓✓✓")
    print(f"File size: {size_gb:.2f} GB")
    
    # Quick check
    df_check = pd.read_pickle(pickle_path)
    print(f"Papers: {len(df_check):,}")
    print(f"Columns: {list(df_check.columns)}")
    print(f"\nSample cleaned abstract:")
    print(df_check['abstract_clean'].iloc[0][:300])
else:
    print("x Not saved yet")

✓✓✓ Success! ✓✓✓
File size: 4.11 GB
Papers: 2,384,617
Columns: ['id', 'title', 'abstract', 'year', 'primary_category', 'all_categories', 'top_level_domain', 'num_categories', 'is_multi_category', 'has_journal', 'num_authors', 'abstract_length', 'title_length', 'abstract_clean']

Sample cleaned abstract:
differential calculation perturbative quantum chromodynamics production massive photon pair hadron collider next lead order perturbative contribution quark antiquark gluon anti quark gluon gluon subprocesses order resummation initial state gluon radiation valid next next lead logarithmic accuracy re


In [14]:
# quick clean up

# keep what we need for clustering and analysis
# using keep rather than drop to keep overview of current columns

keep_columns = [
    'id',                    # paper identifier
    'title',                 # original title (for display in analysis)
    'abstract_clean',        # clean abstract (for TF-IDF) - MAIN FEATURE
    'year',                  # temporal analysis
    'primary_category',      # main category
    'all_categories',        # full category list
    'top_level_domain',      # cs, math, physics, etc.
    'num_categories',        # how many categories
    'is_multi_category',     # multi-category flag
    'has_journal',           # published or preprint (quality signal)
    'num_authors',           # collaboration size
    'abstract_length',       # original length (before cleaning)
    'title_length'           # original title length
]

df_check = pd.read_pickle('data/processed/arxiv_text_cleaned.pkl')
print(f"Before: {df_check.shape}")
print(f"Columns before: {list(df_check.columns)}")

# Keep only needed columns
df_final = df_check[keep_columns].copy()

print(f"\nAfter: {df_final.shape}")
print(f"Columns after: {list(df_final.columns)}")
print(f"Removed {len(df_check.columns) - len(keep_columns)} columns")
print(f"Memory: {df_final.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

# Save final version
df_final.to_pickle('data/processed/arxiv_text_cleaned.pkl')
print("\n✓ Saved optimized version")

Before: (2384617, 14)
Columns before: ['id', 'title', 'abstract', 'year', 'primary_category', 'all_categories', 'top_level_domain', 'num_categories', 'is_multi_category', 'has_journal', 'num_authors', 'abstract_length', 'title_length', 'abstract_clean']

After: (2384617, 13)
Columns after: ['id', 'title', 'abstract_clean', 'year', 'primary_category', 'all_categories', 'top_level_domain', 'num_categories', 'is_multi_category', 'has_journal', 'num_authors', 'abstract_length', 'title_length']
Removed 1 columns
Memory: 2.55 GB

✓ Saved optimized version
